# Importamos las Librerías y Métodos necesarios

In [39]:
import pandas as pd
import numpy as np
import calendar
from sklearn.model_selection import train_test_split
from sklearn import linear_model
from sklearn.metrics import r2_score
from sklearn.tree import DecisionTreeClassifier

# Cargamos los datos

In [23]:
df =pd.read_csv("forestfires.csv", encoding='UTF-8')
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 517 entries, 0 to 516
Data columns (total 13 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   X       517 non-null    int64  
 1   Y       517 non-null    int64  
 2   month   517 non-null    str    
 3   day     517 non-null    str    
 4   FFMC    517 non-null    float64
 5   DMC     517 non-null    float64
 6   DC      517 non-null    float64
 7   ISI     517 non-null    float64
 8   temp    517 non-null    float64
 9   RH      517 non-null    int64  
 10  wind    517 non-null    float64
 11  rain    517 non-null    float64
 12  area    517 non-null    float64
dtypes: float64(8), int64(3), str(2)
memory usage: 52.6 KB


# Analizamos la salud del dataset

In [4]:
def perfilado_inicial(df):
    """Genera un reporte rápido sobre la salud del DataFrame."""
    reporte = pd.DataFrame({
        "Tipo_Dato": df.dtypes,
        "Valores_Nulos": df.isna().sum(),
        "%_Nulos": (df.isna().sum() / len(df) * 100).round(2),
        "Valores_Unicos": df.nunique(),
        "Cardinalidad_%": (df.nunique() / len(df) * 100).round(2)
    })
    return reporte

print(perfilado_inicial(df))

      Tipo_Dato  Valores_Nulos  %_Nulos  Valores_Unicos  Cardinalidad_%
X         int64              0      0.0               9            1.74
Y         int64              0      0.0               7            1.35
month       str              0      0.0              12            2.32
day         str              0      0.0               7            1.35
FFMC    float64              0      0.0             106           20.50
DMC     float64              0      0.0             215           41.59
DC      float64              0      0.0             219           42.36
ISI     float64              0      0.0             119           23.02
temp    float64              0      0.0             192           37.14
RH        int64              0      0.0              75           14.51
wind    float64              0      0.0              21            4.06
rain    float64              0      0.0               7            1.35
area    float64              0      0.0             251         

In [9]:
# Calculamos las estadísticas descriptivas básicas

variables = df.select_dtypes(include='number').columns.values

for variable in variables:
    
    vname = str(variable)
    print('Estadísticos para la variable: ', vname)
    print(df[vname].describe().T, end="\n\n")

Estadísticos para la variable:  X
count    517.000000
mean       4.669246
std        2.313778
min        1.000000
25%        3.000000
50%        4.000000
75%        7.000000
max        9.000000
Name: X, dtype: float64

Estadísticos para la variable:  Y
count    517.000000
mean       4.299807
std        1.229900
min        2.000000
25%        4.000000
50%        4.000000
75%        5.000000
max        9.000000
Name: Y, dtype: float64

Estadísticos para la variable:  FFMC
count    517.000000
mean      90.644681
std        5.520111
min       18.700000
25%       90.200000
50%       91.600000
75%       92.900000
max       96.200000
Name: FFMC, dtype: float64

Estadísticos para la variable:  DMC
count    517.000000
mean     110.872340
std       64.046482
min        1.100000
25%       68.600000
50%      108.300000
75%      142.400000
max      291.300000
Name: DMC, dtype: float64

Estadísticos para la variable:  DC
count    517.000000
mean     547.940039
std      248.066192
min        7.900000

# Realizamos Ingeniería de Características

## Actuaciones a realizar:

1. Transformación logarítmica *(log10)* sobre las variables numéricas que presentan un rango (recorrido) muy grande
2. Tranformación a enteros de las variables *month* y *day*
3. Transformación trigonométrica sobre las variables cíclicas (*month* y *day*)
4. Creación de variable booleana *weekend*
5. Creación de variable booleana *fire*

In [45]:
# Transformaciones Logarítmicas

df["FFMC_Log"] = np.log10(df["FFMC"]+1)
df["DMC_Log"] = np.log10(df['DMC']+1)
df["DC_Log"] = np.log10(df['DC']+1)
df["ISI_Log"] = np.log10(df['ISI']+1)
df["temp_Log"] = np.log10(df['temp']+1)
df["RH_Log"] = np.log10(df['RH']+1)
df["area_Log"] = np.log10(df['area']+1)

# Tranformación a enteros de las variables month y day
meses_map = {v.lower(): k for k, v in enumerate(calendar.month_abbr) if v}
df["month_int"] = df['month'].str.lower().map(meses_map)
dias_map = {v.lower(): k for k, v in enumerate(calendar.day_abbr) if v}
df["day_int"] = df['day'].str.lower().map(dias_map)


# Transformación Trigonométrica de las variables month_int y day_int
df["month_cos"] = np.cos(2 * np.pi * df['month_int'] / 12)
df["day_cos"] = np.cos(2 * np.pi * df['day_int'] / 7)

# Creación de variables Booleanas
df["weekend"] = (df['day_int'] >= 6).astype(int)
df['fire'] = (df['area_Log'] > 0).astype(int)

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 517 entries, 0 to 516
Data columns (total 26 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   X          517 non-null    int64  
 1   Y          517 non-null    int64  
 2   month      517 non-null    str    
 3   day        517 non-null    str    
 4   FFMC       517 non-null    float64
 5   DMC        517 non-null    float64
 6   DC         517 non-null    float64
 7   ISI        517 non-null    float64
 8   temp       517 non-null    float64
 9   RH         517 non-null    int64  
 10  wind       517 non-null    float64
 11  rain       517 non-null    float64
 12  area       517 non-null    float64
 13  FFMC_Log   517 non-null    float64
 14  DMC_Log    517 non-null    float64
 15  DC_Log     517 non-null    float64
 16  ISI_Log    517 non-null    float64
 17  temp_Log   517 non-null    float64
 18  RH_Log     517 non-null    float64
 19  area_Log   517 non-null    float64
 20  month_int  517 non-nu

In [67]:
# Filtrado del DataFrame para eliminar las variables redundantes

dfw = df.drop(columns=['month', 'day', 'FFMC', 'DMC', 'DC', 'ISI', 'temp', 'RH', 'area', 'month_int', 'day_int']).reset_index()
dfw.info()

<class 'pandas.DataFrame'>
RangeIndex: 517 entries, 0 to 516
Data columns (total 16 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   index      517 non-null    int64  
 1   X          517 non-null    int64  
 2   Y          517 non-null    int64  
 3   wind       517 non-null    float64
 4   rain       517 non-null    float64
 5   FFMC_Log   517 non-null    float64
 6   DMC_Log    517 non-null    float64
 7   DC_Log     517 non-null    float64
 8   ISI_Log    517 non-null    float64
 9   temp_Log   517 non-null    float64
 10  RH_Log     517 non-null    float64
 11  area_Log   517 non-null    float64
 12  month_cos  517 non-null    float64
 13  day_cos    517 non-null    float64
 14  weekend    517 non-null    int64  
 15  fire       517 non-null    int64  
dtypes: float64(11), int64(5)
memory usage: 64.8 KB


In [68]:
# Entrenamos un algoritmo de regresión para intentar predecir el area_log a partir de las variables climáticas

X = dfw.drop(columns=['index', 'area_Log', 'fire']).reset_index()
y = dfw['area_Log']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42
)

# Entrenamos el algoritmo Lineal
regresor_lineal = linear_model.LinearRegression()
regresor_lineal.fit(X_train, y_train)

# Hacemos predicciones
y_pred = regresor_lineal.predict(X_test)

score = regresor_lineal.score(X_test, y_test)
params = regresor_lineal.get_params(deep=True)

print(score)


0.10821185175484727


In [69]:
# Entrenamos un algoritmo de clasifiación previamente y pasamos los resultados al algoritmo de regresión

X = dfw.drop(columns=['index', 'area_Log', 'fire']).reset_index()
y_clas = dfw['fire']

X_train, X_test, y_train, y_test = train_test_split(
    X, y_clas, 
    test_size=0.2,
    stratify=y_clas, 
    random_state=42
)

# Entrenamos un Árbol
modelo_arbol = DecisionTreeClassifier(
    criterion="gini",
    max_depth=4,              # Profundidad máxima del árbol. Cámbialo luego a 10
    min_samples_split=10,     # Mínimo de muestras para dividir un nodo. Cambialo luego a 6
    min_samples_leaf=5,       # Mínimo de muestras en nodo hoja. Cambialo luego a 3
    random_state=42
)

modelo_arbol.fit(X_train, y_train)
dfw["fire_pred"] = modelo_arbol.predict(X)

dfw.info()

<class 'pandas.DataFrame'>
RangeIndex: 517 entries, 0 to 516
Data columns (total 17 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   index      517 non-null    int64  
 1   X          517 non-null    int64  
 2   Y          517 non-null    int64  
 3   wind       517 non-null    float64
 4   rain       517 non-null    float64
 5   FFMC_Log   517 non-null    float64
 6   DMC_Log    517 non-null    float64
 7   DC_Log     517 non-null    float64
 8   ISI_Log    517 non-null    float64
 9   temp_Log   517 non-null    float64
 10  RH_Log     517 non-null    float64
 11  area_Log   517 non-null    float64
 12  month_cos  517 non-null    float64
 13  day_cos    517 non-null    float64
 14  weekend    517 non-null    int64  
 15  fire       517 non-null    int64  
 16  fire_pred  517 non-null    int64  
dtypes: float64(11), int64(6)
memory usage: 68.8 KB


In [70]:
# Repetimos las predicciones

X = dfw.drop(columns=['index', 'area_Log', 'fire']).reset_index()
y = dfw['area_Log']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42
)

# Entrenamos el algoritmo Lineal
regresor_lineal = linear_model.LinearRegression()
regresor_lineal.fit(X_train, y_train)

# Hacemos predicciones
y_pred = regresor_lineal.predict(X_test)

score = regresor_lineal.score(X_test, y_test)
params = regresor_lineal.get_params(deep=True)

print(score)

0.34897104090572606
